# SCD type 1

In [0]:
%sql
DROP TABLE IF EXISTS datamodeling.default.scdtype1_source;
CREATE TABLE IF NOT EXISTS datamodeling.default.scdtype1_source
(
  pro_id INT,
  pro_name STRING,
  pro_cat STRING,
  processDate DATE
)

In [0]:
%sql
INSERT INTO datamodeling.default.scdtype1_source
VALUES
(1,'Product1','Category1',current_date()),
(2,'Product2','Category2',current_date()),
(3,'Product3','newcatgory',current_date())


num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
SELECT * FROM datamodeling.default.scdtype1_source

pro_id,pro_name,pro_cat,processDate
1,Product1,Category1,2025-11-05
2,Product2,Category2,2025-11-05
3,Product3,newcatgory,2025-11-05


In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.gold.scdtype1_table
(
  pro_id INT,
  pro_name STRING,
  pro_cat STRING,
  processDate DATE
)


In [0]:
spark.sql("Select * from datamodeling.default.scdtype1_source").createOrReplaceTempView("src")

In [0]:
%sql
SELECT * FROM src

pro_id,pro_name,pro_cat,processDate
1,Product1,Category1,2025-11-05
2,Product2,Category2,2025-11-05
3,Product3,newcatgory,2025-11-05


In [0]:
%sql
MERGE INTO datamodeling.gold.scdtype1_table AS trg
USING src
ON trg.pro_id = src.pro_id
WHEN MATCHED AND src.processDate >= trg.processDate THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,3,0,0


In [0]:
%sql
SELECT * FROM datamodeling.gold.scdtype1_table

pro_id,pro_name,pro_cat,processDate
1,Product1,Category1,2025-11-05
2,Product2,Category2,2025-11-05
3,Product3,newcatgory,2025-11-05


# SCD Type 2


In [0]:
%sql
DROP TABLE IF EXISTS datamodeling.default.scdtype2_source;
CREATE TABLE IF NOT EXISTS datamodeling.default.scdtype2_source
(
  pro_id INT,
  pro_name STRING,
  pro_cat STRING,
  processDate DATE
)

In [0]:
%sql
INSERT INTO datamodeling.default.scdtype2_source
VALUES
(1,'Product1','Category1',current_date()),
(2,'Product2','Category2',current_date()),
(3,'Product3','newcategory',current_date())

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.gold.scdtype2_table
(
  pro_id INT,
  pro_name STRING,
  pro_cat STRING,
  processDate DATE,
  start_date DATE,
  end_date DATE,
  is_current STRING
)

In [0]:
%sql
SELECT *,
    current_timestamp() as start_date,
    CAST('3000-01-01' AS timestamp) as end_date,
    'Y' as is_current
FROM datamodeling.default.scdtype2_source

pro_id,pro_name,pro_cat,processDate,start_date,end_date,is_current
1,Product1,Category1,2025-11-05,2025-11-05T11:21:10.476686Z,3000-01-01T00:00:00Z,Y
2,Product2,Category2,2025-11-05,2025-11-05T11:21:10.476686Z,3000-01-01T00:00:00Z,Y
3,Product3,newcategory,2025-11-05,2025-11-05T11:21:10.476686Z,3000-01-01T00:00:00Z,Y


In [0]:
spark.sql(
    """
    SELECT *,
        current_timestamp() as start_date,
        CAST('3000-01-01' AS timestamp) as end_date,
        'Y' as is_current
    FROM datamodeling.default.scdtype2_source
    """
).createOrReplaceTempView("srctype2")

In [0]:
%sql
SELECT * FROM srctype2

pro_id,pro_name,pro_cat,processDate,start_date,end_date,is_current
1,Product1,Category1,2025-11-05,2025-11-05T11:21:14.434524Z,3000-01-01T00:00:00Z,Y
2,Product2,Category2,2025-11-05,2025-11-05T11:21:14.434524Z,3000-01-01T00:00:00Z,Y
3,Product3,newcategory,2025-11-05,2025-11-05T11:21:14.434524Z,3000-01-01T00:00:00Z,Y


In [0]:
%sql
MERGE INTO datamodeling.gold.scdtype2_table as trg
USING srctype2 
ON srctype2.pro_id = trg.pro_id
AND trg.is_current = 'Y'

-- When we have New Data with Updates
WHEN MATCHED AND (
  srctype2.pro_cat <> trg.pro_cat OR
  srctype2.processDate <> trg.processDate OR
  srctype2.pro_name <> trg.pro_name
) THEN
UPDATE SET
  trg.end_date = current_timestamp(),
  trg.is_current = 'N'


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


In [0]:
%sql
MERGE INTO datamodeling.gold.scdtype2_table as trg
USING srctype2 
ON srctype2.pro_id = trg.pro_id
AND trg.is_current = 'Y'

WHEN NOT MATCHED THEN INSERT(
  pro_id,
  pro_name,
  pro_cat,
  processDate,
  start_date,
  end_date,
  is_current
) VALUES (
  srctype2.pro_id,
  srctype2.pro_name,
  srctype2.pro_cat,
  srctype2.processDate,
  srctype2.start_date,
  srctype2.end_date,
  srctype2.is_current
)
  

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,0,0,1


In [0]:
%sql
SELECT * FROM datamodeling.gold.scdtype2_table

pro_id,pro_name,pro_cat,processDate,start_date,end_date,is_current
1,Product1,Category1,2025-11-05,2025-11-05,3000-01-01,Y
2,Product2,Category2,2025-11-05,2025-11-05,3000-01-01,Y
3,Product3,Category3,2025-11-05,2025-11-05,2025-11-05,N
3,Product3,newcategory,2025-11-05,2025-11-05,3000-01-01,Y
